# Modeling Human Activity States Using Hidden Markov Models

**Human Activity Recognition (HAR) from smartphone accelerometer & gyroscope signals.**

This notebook implements the full pipeline:
1. Load raw sensor recordings (CSV)
2. Extract time- & frequency-domain features (sliding windows)
3. Define & train a Gaussian HMM (Baum–Welch)
4. Decode the most likely activity sequence (Viterbi)
5. Visualize the transition matrix and decoded sequences
6. Evaluate on unseen data (sensitivity / specificity / accuracy)

> **IMPORTANT:** The `data/` folder currently holds *synthetic placeholder*
> recordings so the notebook runs end-to-end. **Replace them with your real
> Sensor Logger exports** (same columns, same filename convention
> `activity_NN.csv`, e.g. `walking_00.csv`) and re-run all cells.


## 0. Setup

In [ ]:
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.fft import rfft, rfftfreq
from hmmlearn import hmm

np.random.seed(42)
plt.rcParams['figure.dpi'] = 110

# ---- CONFIG: set to YOUR phone's sampling rate (Hz) ----
SAMPLING_RATE = 50      # e.g. 50 Hz. If members used different rates, resample first.
WINDOW_SEC    = 1.0     # window length in seconds
OVERLAP       = 0.5     # 50% overlap between consecutive windows

ACTIVITIES = ["still", "standing", "walking", "jumping"]
ACT2ID = {a: i for i, a in enumerate(ACTIVITIES)}
print("Activities:", ACT2ID)

## 1. Data Collection

Recordings were made with the **Sensor Logger** app. Each `.csv` contains a
`timestamp` column plus 3-axis accelerometer and 3-axis gyroscope:

`timestamp, acc_x, acc_y, acc_z, gyro_x, gyro_y, gyro_z`

Filenames encode the activity label: `walking_00.csv`, `jumping_03.csv`, etc.

> **Sampling rate:** recordings here come from a single device at a fixed rate.
> If a recording's rate ever differs, use the `resample_to` helper below to bring
> it to a common `SAMPLING_RATE` before feature extraction.

In [ ]:
def load_csv(path):
    df = pd.read_csv(path)
    needed = ["acc_x","acc_y","acc_z","gyro_x","gyro_y","gyro_z"]
    missing = [c for c in needed if c not in df.columns]
    if missing:
        raise ValueError(f"{os.path.basename(path)} missing columns {missing}. "
                         f"Found {list(df.columns)}")
    return df

def label_from_filename(path):
    base = os.path.basename(path).lower()
    for a in ACTIVITIES:
        if base.startswith(a):
            return a
    raise ValueError(f"Cannot infer activity from '{base}'")

def resample_to(df, src_fs, dst_fs):
    """Optional: linear resample a recording from src_fs to dst_fs."""
    if src_fs == dst_fs:
        return df
    n_new = int(len(df) * dst_fs / src_fs)
    t_old = np.linspace(0, 1, len(df))
    t_new = np.linspace(0, 1, n_new)
    out = {"timestamp": np.arange(n_new)/dst_fs}
    for c in ["acc_x","acc_y","acc_z","gyro_x","gyro_y","gyro_z"]:
        out[c] = np.interp(t_new, t_old, df[c].values)
    return pd.DataFrame(out)

files = sorted(glob.glob("data/*.csv"))
print(f"Found {len(files)} training recordings")
example = load_csv(files[0])
print("Example:", os.path.basename(files[0]), example.shape)
example.head()

### Quick look at the raw signals

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 6))
for ax, act in zip(axes.ravel(), ACTIVITIES):
    path = sorted(glob.glob(f"data/{act}_*.csv"))[0]
    d = load_csv(path)
    t = np.arange(len(d)) / SAMPLING_RATE
    ax.plot(t, d.acc_x, lw=.7, label="acc_x")
    ax.plot(t, d.acc_y, lw=.7, label="acc_y")
    ax.plot(t, d.acc_z, lw=.7, label="acc_z")
    ax.set_title(act); ax.set_xlabel("s"); ax.set_ylabel("m/s²")
axes[0,0].legend(fontsize=7, ncol=3)
plt.tight_layout(); plt.savefig("figures/raw_signals.png", bbox_inches="tight"); plt.show()

## 2. Feature Extraction

Each recording is split into overlapping windows. For every window we compute:

**Time-domain:** per-axis mean, std, variance; signal magnitude area (SMA);
resultant-magnitude mean/std; inter-axis correlations (xy, xz, yz).

**Frequency-domain (on acceleration magnitude via FFT):** dominant frequency,
spectral energy.

The resulting feature vector per window is one **observation** for the HMM.

In [ ]:
def window_features(win, fs):
    feats = {}
    axes = ["ax","ay","az","gx","gy","gz"]
    for j, name in enumerate(axes):
        x = win[:, j]
        feats[f"{name}_mean"] = np.mean(x)
        feats[f"{name}_std"]  = np.std(x)
        feats[f"{name}_var"]  = np.var(x)
    acc = win[:, 0:3]
    feats["acc_sma"] = np.mean(np.sum(np.abs(acc), axis=1))
    acc_mag = np.linalg.norm(acc, axis=1)
    feats["acc_mag_mean"] = np.mean(acc_mag)
    feats["acc_mag_std"]  = np.std(acc_mag)
    def corr(a,b):
        if np.std(a)<1e-8 or np.std(b)<1e-8: return 0.0
        return np.corrcoef(a,b)[0,1]
    feats["corr_xy"] = corr(acc[:,0], acc[:,1])
    feats["corr_xz"] = corr(acc[:,0], acc[:,2])
    feats["corr_yz"] = corr(acc[:,1], acc[:,2])
    # frequency domain on detrended acc magnitude
    n = len(acc_mag)
    detr = acc_mag - acc_mag.mean()
    fftv = np.abs(rfft(detr)); freqs = rfftfreq(n, 1.0/fs)
    if len(fftv) > 1:
        k = np.argmax(fftv[1:]) + 1
        feats["dom_freq"] = freqs[k]
        feats["spectral_energy"] = np.sum(fftv**2)/n
    else:
        feats["dom_freq"] = 0.0; feats["spectral_energy"] = 0.0
    return feats

def extract_features(df, fs, label=None):
    data = df[["acc_x","acc_y","acc_z","gyro_x","gyro_y","gyro_z"]].values
    win_n = int(WINDOW_SEC*fs); step = max(1, int(win_n*(1-OVERLAP)))
    rows = []
    for s in range(0, len(data)-win_n+1, step):
        f = window_features(data[s:s+win_n], fs)
        if label is not None:
            f["label"] = label; f["label_id"] = ACT2ID[label]
        rows.append(f)
    return pd.DataFrame(rows)

def build_dataset(folder, fs):
    feats, seqs = [], []
    for path in sorted(glob.glob(os.path.join(folder,"*.csv"))):
        fdf = extract_features(load_csv(path), fs, label_from_filename(path))
        if len(fdf):
            feats.append(fdf); seqs.append(fdf)
    return pd.concat(feats, ignore_index=True), seqs

full, sequences = build_dataset("data", SAMPLING_RATE)
feat_cols = [c for c in full.columns if c not in ("label","label_id")]
print("Feature matrix:", full[feat_cols].shape, "| windows per class:")
print(full["label"].value_counts())
full.head()

## 3. Define Model Components

| Element | In this project |
|---|---|
| **Hidden states (Z)** | the 4 activities: still, standing, walking, jumping |
| **Observations (X)** | the per-window feature vectors above |
| **Transition probs (A)** | learned P(activityₜ → activityₜ₊₁) |
| **Emission probs (B)** | per-state multivariate Gaussian over features |
| **Initial probs (π)** | learned P(starting activity) |

We use a **Gaussian HMM** (continuous emissions) with a diagonal covariance,
trained by **Baum–Welch** (EM). Features are standardized first.

In [ ]:
X = full[feat_cols].values
mu, sd = X.mean(0), X.std(0) + 1e-8      # save mu/sd to standardize unseen data later
Xn = (X - mu) / sd
lengths = [len(s) for s in sequences]     # keep each recording as one sequence
print("Total windows:", len(Xn), "| sequences:", len(lengths))

## 4. Model Implementation

### 4a. Baum–Welch training (via hmmlearn)
`hmmlearn`'s `GaussianHMM.fit` runs the Baum–Welch (EM) algorithm to estimate
A, B and π.

In [ ]:
model = hmm.GaussianHMM(n_components=4, covariance_type="diag",
                        n_iter=200, tol=1e-4, random_state=42)
model.fit(Xn, lengths)
print("Converged log-likelihood:", round(model.monitor_.history[-1], 2))
print("Iterations:", len(model.monitor_.history))

### 4b. Viterbi decoding
`model.predict` uses the **Viterbi algorithm** to return the single most likely
hidden-state sequence. Because HMM state indices are arbitrary, we align learned
states to true activity labels by the best-matching permutation.

In [ ]:
from itertools import permutations
def align_states(pred, true, n=4):
    best, bacc = None, -1
    for p in permutations(range(n)):
        acc = np.mean([p[s] for s in pred] == true)
        if acc > bacc: bacc, best = acc, p
    return np.array([best[s] for s in pred]), best

states = model.predict(Xn, lengths)
mapped, perm = align_states(states, full["label_id"].values)
train_acc = np.mean(mapped == full["label_id"].values)
print(f"State->label permutation: {perm}")
print(f"Training accuracy: {train_acc:.3f}")

### 4c. (Optional) From-scratch Viterbi

To satisfy the "implement Viterbi" requirement explicitly, here is a pure-NumPy
Viterbi that decodes using the parameters `hmmlearn` learned. It should match
`model.predict`.

In [ ]:
def viterbi(log_startprob, log_transmat, framelogprob):
    n_obs, n_states = framelogprob.shape
    delta = np.zeros((n_obs, n_states)); psi = np.zeros((n_obs, n_states), int)
    delta[0] = log_startprob + framelogprob[0]
    for t in range(1, n_obs):
        for j in range(n_states):
            seq = delta[t-1] + log_transmat[:, j]
            psi[t, j] = np.argmax(seq)
            delta[t, j] = np.max(seq) + framelogprob[t, j]
    path = np.zeros(n_obs, int); path[-1] = np.argmax(delta[-1])
    for t in range(n_obs-2, -1, -1):
        path[t] = psi[t+1, path[t+1]]
    return path

# decode the first recording with our scratch implementation
seq0 = Xn[:lengths[0]]
flp = model._compute_log_likelihood(seq0)
scratch = viterbi(np.log(model.startprob_+1e-12), np.log(model.transmat_+1e-12), flp)
builtin = model.predict(seq0)
print("Scratch Viterbi matches hmmlearn:", np.array_equal(scratch, builtin))

## 5. Visualization
### 5a. Transition matrix heatmap

In [ ]:
A = model.transmat_
# reorder rows/cols to activity order using the permutation
inv = np.argsort(perm)
A_ord = A[np.ix_(inv, inv)]
fig, ax = plt.subplots(figsize=(5.5,4.5))
im = ax.imshow(A_ord, cmap="viridis", vmin=0, vmax=1)
ax.set_xticks(range(4)); ax.set_yticks(range(4))
ax.set_xticklabels(ACTIVITIES, rotation=45, ha="right"); ax.set_yticklabels(ACTIVITIES)
for i in range(4):
    for j in range(4):
        ax.text(j, i, f"{A_ord[i,j]:.2f}", ha="center", va="center",
                color="white" if A_ord[i,j]<.6 else "black", fontsize=9)
ax.set_title("Transition matrix  P(row → col)")
ax.set_xlabel("to"); ax.set_ylabel("from")
plt.colorbar(im, fraction=.046)
plt.tight_layout(); plt.savefig("figures/transition_matrix.png", bbox_inches="tight"); plt.show()

### 5b. Decoded sequence vs. ground truth

In [ ]:
fig, ax = plt.subplots(figsize=(12,3.2))
ax.plot(full["label_id"].values, "o-", ms=3, lw=.8, label="true", alpha=.7)
ax.plot(mapped, "x--", ms=4, lw=.8, label="decoded (Viterbi)", alpha=.7)
ax.set_yticks(range(4)); ax.set_yticklabels(ACTIVITIES)
ax.set_xlabel("window index (all recordings concatenated)")
ax.set_title("Decoded activity sequence vs ground truth"); ax.legend()
plt.tight_layout(); plt.savefig("figures/decoded_sequence.png", bbox_inches="tight"); plt.show()

## 6. Model Evaluation with Unseen Data

We evaluate on a separate session in `data/unseen/` (new recordings not used in
training). Features are standardized with the **training** `mu`/`sd`, and states
mapped with the **training** permutation — no peeking.

In [ ]:
def per_class_metrics(y_true, y_pred, n=4):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    out = {}
    for s in range(n):
        tp = np.sum((y_pred==s)&(y_true==s)); fn = np.sum((y_pred!=s)&(y_true==s))
        tn = np.sum((y_pred!=s)&(y_true!=s)); fp = np.sum((y_pred==s)&(y_true!=s))
        out[s] = dict(
            n=int(np.sum(y_true==s)),
            sensitivity = tp/(tp+fn) if (tp+fn) else 0.0,
            specificity = tn/(tn+fp) if (tn+fp) else 0.0,
            accuracy    = (tp+tn)/len(y_true))
    return out

if os.path.isdir("data/unseen") and glob.glob("data/unseen/*.csv"):
    uf, useqs = build_dataset("data/unseen", SAMPLING_RATE)
    Xu = (uf[feat_cols].values - mu) / sd
    ust = model.predict(Xu, [len(s) for s in useqs])
    umapped = np.array([perm[s] for s in ust])
    uacc = np.mean(umapped == uf["label_id"].values)
    print(f"UNSEEN overall accuracy: {uacc:.3f}\n")
    metrics = per_class_metrics(uf["label_id"].values, umapped)
    tbl = pd.DataFrame([
        {"State (Activity)": ACTIVITIES[s], "Number of Samples": d["n"],
         "Sensitivity": round(d["sensitivity"],3),
         "Specificity": round(d["specificity"],3),
         "Overall Accuracy": round(d["accuracy"],3)}
        for s,d in metrics.items()])
    display(tbl)
    tbl.to_csv("figures/unseen_metrics.csv", index=False)
else:
    print("No unseen data found in data/unseen/")

### 6b. Confusion matrix on unseen data

In [ ]:
from itertools import product
cm = np.zeros((4,4), int)
for t,p in zip(uf["label_id"].values, umapped):
    cm[t,p]+=1
fig, ax = plt.subplots(figsize=(5,4.2))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(4)); ax.set_yticks(range(4))
ax.set_xticklabels(ACTIVITIES, rotation=45, ha="right"); ax.set_yticklabels(ACTIVITIES)
for i,j in product(range(4),range(4)):
    ax.text(j,i,cm[i,j],ha="center",va="center",
            color="white" if cm[i,j]>cm.max()/2 else "black")
ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title("Confusion matrix (unseen)")
plt.colorbar(im, fraction=.046)
plt.tight_layout(); plt.savefig("figures/confusion_unseen.png", bbox_inches="tight"); plt.show()

## 7. Analysis and Reflection

*Fill this in with observations from **your** data.* Prompts:

- **Easiest / hardest to distinguish:** Typically *still* vs *standing* are the
  hardest pair (both low-motion; separated mainly by variance / sway), while
  *jumping* is easiest (huge vertical spikes, high spectral energy). *Walking*
  has a clear ~2 Hz dominant frequency.
- **Transition probabilities:** Diagonal dominance (high self-transition) means
  activities persist across adjacent windows — realistic, since people don't
  switch activity every 0.5 s. Comment on any large off-diagonal values.
- **Noise / sampling rate:** Lower sampling rates blur the frequency features
  (dominant frequency, spectral energy) and can merge walking/jumping. Sensor
  noise inflates variance for still/standing and hurts their separation.
- **Improvements:** more data per class, additional features (jerk, entropy,
  axis-wise FFT bands), a magnetometer, longer windows, or a supervised
  classifier baseline to compare against the HMM.

## 8. Conclusion

Summarize: did the HMM recover meaningful states? How well did it generalize to
unseen data (cite the table)? What is the single biggest limitation and the most
promising next step?
